# InStudy 2.0 — Project Progress Notebook


---
This notebook documents the full development journey of **InStudy 2.0**, an AI-powered study assistant built for university and pretty much all students. It covers what was built, when it was built, how everything connects, and the current state of the system.

---

## 1. What Is InStudy 2.0?

InStudy 2.0 is a personal AI study assistant. A student uploads their study materials (PDFs, Word documents, text files) and the platform helps them study through:

- **AI Tutor** — ask questions about your documents and get structured answers
- **Flashcards** — auto-generated question/answer cards from your materials
- **Quizzes** — practice tests generated from your documents
- **Summaries** — condensed versions of your study material
- **Study Planner** — a personalised study schedule based on your exam date
- **Dashboard** — tracks study hours, questions asked, and progress over time
- **Document Viewer** — read uploaded files directly inside the platform and add personal annotations
- **Mastery Tracker** — monitors which concepts have been learned over time

The system runs **fully locally** — no internet connection needed after setup. The AI model runs on the user's own machine using Ollama.

---

## 2. Technology Stack

| Layer | Technology | Purpose |
|---|---|---|
| Frontend (UI) | Streamlit (Python) | The web interface the user sees |
| Backend (API) | FastAPI (Python) | Handles all logic and data |
| AI Model | Ollama (local LLM) | Generates answers, quizzes, summaries |
| Embeddings | Sentence Transformers | Converts text to searchable vectors |
| Vector Search | FAISS | Finds relevant document sections fast |
| User Database | SQLite | Stores user accounts and sessions |
| PDF Rendering | PyMuPDF (fitz) | Renders PDF pages as images |
| Document Processing | LangChain, pdfplumber, python-docx | Extracts and chunks text from files |

---

## 3. Project Structure

```
InStudy 2.0/
├── backend/                    ← FastAPI server (runs on port 8000)
│   ├── main.py                 ← Entry point, registers all routes
│   ├── config.py               ← Settings (model name, chunk size, paths)
│   ├── requirements.txt        ← Python packages needed
│   ├── api/routes/             ← API endpoints (one file per feature)
│   │   ├── auth.py             ← Login / Register
│   │   ├── documents.py        ← Upload, view, annotate, render documents
│   │   ├── chat.py             ← AI Tutor Q&A
│   │   ├── quiz.py             ← Quiz generation and scoring
│   │   ├── flashcards.py       ← Flashcard generation
│   │   ├── summary.py          ← Summary generation
│   │   ├── planner.py          ← Study plan creation
│   │   ├── stats.py            ← User activity tracking
│   │   ├── mastery.py          ← Learning progress tracking
│   │   └── admin.py            ← Admin controls
│   ├── services/               ← Business logic (one file per feature)
│   ├── database/               ← SQLite database access
│   ├── models/                 ← Data shapes (request/response formats)
│   ├── middleware/             ← Token verification on every request
│   └── uploads/                ← Stored user documents
│
├── frontend/                   ← Streamlit UI (runs on port 8501)
│   ├── app.py                  ← Main app, navigation bar, routing, theme
│   ├── pages/                  ← One file per page/feature
│   ├── components/
│   │   ├── auth_guard.py       ← Login/signup forms
│   │   └── document_viewer.py  ← Inline document viewer + annotations
│   └── utils/                  ← Helper functions (auth, API calls, UI)
│
└── docs/                       ← Documentation files
```

---

## 4. How Everything Connects

```
Student opens browser
        ↓
Streamlit Frontend  (localhost:8501)
        ↓  HTTP requests with auth token
FastAPI Backend     (localhost:8000)
        ↓
   ┌──────────────────────────────────────┐
   │  Document uploaded?                  │
   │  → Extract text → Chunk → Embed      │
   │  → Store in FAISS vector store       │
   └──────────────────────────────────────┘
        ↓
   ┌──────────────────────────────────────┐
   │  Question asked?                     │
   │  → Search FAISS for relevant text    │
   │  → Send text + question to Ollama    │
   │  → Return structured answer          │
   └──────────────────────────────────────┘
        ↓
Ollama LLM          (localhost:11434) — runs locally
```

Every page in the frontend talks to the backend through HTTP. The backend handles all AI logic, file storage, and database operations. The frontend only handles display and user interaction.

---

## 5. Development Timeline

### Phase 1 — Foundation (March 9, 2026)

**What was built:**
- Core backend with FastAPI
- Document upload and processing pipeline (PDF, DOCX, TXT)
- FAISS vector store for document search
- Hybrid RAG system for AI Q&A
- Streamlit frontend with AI Tutor, Flashcards, Quiz, Summary, Planner pages
- Real-time streaming responses in the AI Tutor (word-by-word display)
- Dynamic dashboard with activity tracking
- Progress indicators on Quiz, Summary, and Flashcard generation

**Key design decision — Hybrid RAG:**
The AI uses two paths:
- If the document has relevant content → use that content + AI
- If not → use AI general knowledge alone

This means students always get an answer, even if their materials don't cover the topic.

**AI answer structure (every response follows this format):**
1. Concept definition
2. Step-by-step explanation
3. Example
4. Possible exam question
5. Quick summary

---

### Phase 2 — Authentication & Admin System (March 10, 2026)

**What was built:**
- Full user registration and login system
- SQLite database for user accounts and sessions
- JWT token-based authentication (every API request is verified)
- Session management with 90-day expiry
- Admin panel — view users, delete accounts, promote to admin
- Default admin account created on first run
- All user data is isolated — one user cannot see another's documents

**How auth works:**
```
User logs in → Backend verifies password → Issues JWT token
Token stored in browser session
Every request sends token in header → Middleware verifies it
If invalid → Request rejected
```

---

### Phase 3 — Page-Aware AI Tutor & Quiz Fixes (March 12, 2026)

**What was built:**
- AI Tutor can now answer questions about specific pages (e.g. "explain page 5")
- Quiz generation made more robust — fixed parsing errors and missing fields
- UX improvements across multiple pages
- Better error handling throughout the backend

---

### Phase 4 — Flashcard Images & Admin Navigation (March 20, 2026)

**What was built:**
- Flashcards now include relevant images from a local image library
- Images matched to flashcard topics (science, math, technology, etc.)
- Enhanced explanation levels on flashcards
- Admin navigation bug fixed
- All documentation moved into a dedicated `docs/` folder

---

### Phase 5 — Document Viewer & Annotations (March 24–25, 2026)

**What was built:**
- Inline document viewer added to all pages (AI Tutor, Flashcards, Quiz, Summary, Planner)
- Two-column layout: document on the left, annotation panel on the right
- Users can add annotations directly to paragraphs in their documents
- Annotation types: Key Point, Explanation, Summary, Note, Question
- Annotations stored as sidecar `.json` files alongside the uploaded document
- Export feature: download an annotated `.docx` file with original content + annotations interleaved
- Duplicate annotation prevention
- Dashboard charts updated to show daily granularity when data spans only 1–2 months
- Glassmorphism dark UI theme applied across the platform

**How annotations work:**
```
User opens document → selects a paragraph → writes annotation → clicks Save
Saved to: uploads/{user_id}/{course_id}/{filename}.annotations.json
On export: original paragraphs + annotations merged into a Word document
```

---

### Phase 6 — PDF Page Rendering, Performance & Mastery (March 25 – April 6, 2026)

**What was built:**
- PDF documents now render as actual page images using PyMuPDF
- Each page rendered at 2x resolution for crisp display
- Page navigation with Prev/Next buttons and a slider
- Server-side page cache — each page is only rendered once per session
- Client-side cache — Streamlit doesn't re-fetch the same page on reruns
- Adjacent pages pre-fetched silently so navigation feels instant
- DOCX and TXT files use a styled HTML viewer with heading and list detection
- Dashboard charts fixed — auto-switches to daily view when all data is within 1–2 months
- Quiz service fixed — LLM failure now handled gracefully with fallback
- Mastery tracking page added
- AI adaptive study planning improvements
- Major UI stabilisation across all pages

---

## 6. Current State of the System (April 2026)

| Feature | Status |
|---|---|
| User registration & login | Working |
| Admin panel | Working |
| Document upload (PDF, DOCX, TXT) | Working |
| AI Tutor with page-aware Q&A | Working |
| Streaming responses | Working |
| ELI12 mode (explain simply) | Working |
| Flashcard generation with images | Working |
| Quiz generation and scoring | Working |
| Summary generation | Working |
| Study Planner | Working |
| Dashboard with activity charts | Working |
| Document viewer (PDF as images) | Working |
| Document annotations | Working |
| Annotated document export (.docx) | Working |
| Mastery tracking | Working |
| Dark glassmorphism UI | Applied |

---

## 7. A Student's Journey Through the Platform

**Step 1 — Register & Login**
```
Student creates account → password hashed and stored in SQLite
Login → JWT token issued → stored in browser session
```

**Step 2 — Create a Course**
```
Student creates a course (e.g. "Machine Learning")
Course stored as a folder: uploads/{user_id}/machine_learning/
```

**Step 3 — Upload a Document**
```
Student uploads ML.pdf
Backend extracts text → splits into 500-character chunks
Each chunk converted to a vector (embedding)
Vectors stored in FAISS: vector_store/{user_id}_machine_learning/
```

**Step 4 — Ask a Question**
```
Student asks: "What is gradient descent?"
Backend searches FAISS for relevant chunks
If found → sends chunks + question to Ollama LLM
LLM returns structured answer streamed word-by-word
Activity logged to activity.json
```

**Step 5 — Generate a Quiz**
```
Student clicks Generate Quiz
Backend retrieves document chunks → LLM generates questions
Student answers → backend scores → results shown
Quiz count logged to activity tracker
```

**Step 6 — View Document & Annotate**
```
Student opens document viewer
PDF rendered page-by-page as images (PyMuPDF)
Student selects a paragraph → adds annotation (e.g. Key Point)
Annotation saved to ML.pdf.annotations.json
Student exports → receives annotated Word document
```

**Step 7 — Check Dashboard**
```
Dashboard reads activity.json
Shows: study hours, questions asked, quizzes taken, course engagement
Charts auto-switch between daily and monthly view based on data range
```

---

## 8. Guide for a New Developer

### How to Run the Project

**Requirements:** Python 3.10+, Ollama installed and running

```bash
# Pull the AI model
ollama pull gemma:2b

# Start the backend
cd backend
pip install -r requirements.txt
uvicorn main:app --reload --port 8000

# Start the frontend (new terminal)
cd frontend
pip install -r requirements.txt
streamlit run app.py
```

**Default admin login:** `admin@instudy.com` / `admin123`

### Where to Find Things

| I want to change... | Look in... |
|---|---|
| How the AI answers questions | `backend/services/rag_service.py` |
| How quizzes are generated | `backend/services/quiz_service.py` |
| How documents are processed | `backend/services/document_processor.py` |
| The document viewer UI | `frontend/components/document_viewer.py` |
| The dashboard charts | `frontend/pages/dashboard.py` |
| API endpoints | `backend/api/routes/` |
| App navigation and theme | `frontend/app.py` |
| Configuration (model, chunk size) | `backend/config.py` |
| User database | `backend/database/auth_db.py` |

### Key Concepts

**FAISS Vector Store** — When a document is uploaded, its text is converted into numbers (vectors). FAISS stores these and quickly finds which parts of a document are most relevant to a question. Each user has their own isolated vector store.

**Hybrid RAG** — The AI first searches the vector store. If it finds something relevant, it uses that as context. If not, it answers from general knowledge. This is why the AI always gives a useful answer.

**Activity Tracking** — Every question asked, quiz taken, and document uploaded is logged to `uploads/{user_id}/activity.json`. The dashboard reads this file to show statistics.

**Annotations** — Stored as `{filename}.annotations.json` next to the uploaded file. Each annotation has a paragraph index, content, type, and timestamp.

---

## 9. Summary

InStudy 2.0 started as a basic document Q&A tool and has grown into a full study platform. Over approximately 4 weeks of development, the following was built:

- A complete authentication system with user isolation
- A hybrid AI tutoring system that always gives useful answers
- Auto-generation of quizzes, flashcards, summaries, and study plans
- A document viewer that renders PDFs as images with page navigation
- An annotation system that lets students enrich their study materials
- A dashboard that tracks study activity over time with smart chart granularity
- A polished dark UI with glassmorphism styling
- An admin panel for user management
- A mastery tracker for learning progress

The system runs entirely locally — no cloud costs, no data leaving the machine — making it suitable for students who want privacy and offline access.

---
*InStudy 2.0 — Built with FastAPI, Streamlit, Ollama, FAISS, and PyMuPDF*